In [1]:
import pandas as pd
import json
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
file_path = BASE_DIR / "data" / "valid_questions.json"

with open(file_path, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

print(json.dumps(questions_data[:2], indent=4, ensure_ascii=False))

[
    {
        "question": "조용한 분위기면서 공간이 넉넉한 카페 추천해 줄 수 있어?",
        "type": "single",
        "keywords": [
            {
                "type": "ATMOSPHERE",
                "keyword": "사이즈"
            }
        ],
        "label": 1
    },
    {
        "question": "자리 넓고 편안한 느낌의 카페 어디 있을까?",
        "type": "single",
        "keywords": [
            {
                "type": "ATMOSPHERE",
                "keyword": "사이즈"
            }
        ],
        "label": 1
    }
]


In [49]:
file_path = BASE_DIR / "data" / "market_item.csv"

df = pd.read_csv(file_path)

In [51]:
df['최종_키워드'].unique()

array(['샌드위치_가격대비 괜찮다', '마카롱_가성비 있다', '아이스아메리카노_양이 많다', ..., '카푸치노',
       '마스카포네케이크', '소금프레첼베이글'], shape=(1425,), dtype=object)

In [52]:
for i in df['키워드타입'].unique():
    res = len(df[df['키워드타입'] == i]['최종_키워드'].unique())
    print(f"키워드 타입: {i}, 개수:  {res}", )

키워드 타입: MENU, 개수:  628
키워드 타입: FACILITY, 개수:  392
키워드 타입: ATMOSPHERE, 개수:  269
키워드 타입: TARGET, 개수:  239


In [60]:
df[df['최종_키워드'] == '편안한 공간']

,사업장명,키워드타입,최종_키워드,tfidf_score,sentiment,count,global_count
1656,그릿 커피랩 신림점,ATMOSPHERE,편안한 공간,0.8066,긍정,NaN,NaN
1771,걸작커피,ATMOSPHERE,편안한 공간,0.6049,긍정,NaN,NaN
2166,커피 브륄레,ATMOSPHERE,편안한 공간,0.3122,긍정,NaN,NaN
2167,에놀라 톤,ATMOSPHERE,편안한 공간,0.3122,긍정,NaN,NaN
2430,백억커피,TARGET,편안한 공간,1.0385,긍정,NaN,NaN


In [57]:
# build_keyword_context 수정
# seed_types 로직 제거 → 항상 3타입 전부 제공
def build_keyword_context(pool: dict) -> str:
    lines = []
    for ktype in ["ATMOSPHERE", "FACILITY", "TARGET"]:
        lines.append(f"[{ktype}]\n" + ", ".join(pool[ktype]))
    return "\n\n".join(lines)

In [61]:
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

batch = client.batches.retrieve("batch_6a1d163b59d081908ab8d6f37fb2316e")
print(batch)

Batch(id='batch_6a1d163b59d081908ab8d6f37fb2316e', completion_window='24h', created_at=1780291131, endpoint='/v1/chat/completions', input_file_id='file-2z4nKdkJaCYiyaTKrR8Whc', object='batch', status='failed', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=Errors(data=[BatchError(code='token_limit_exceeded', line=None, message='Enqueued token limit reached for gpt-4.1-mini in organization org-xHgIkArPtViBvUQGZwBMf9IT. Limit: 2,000,000 enqueued tokens. Please try again once some in_progress batches have been completed.', param=None)], object='list'), expired_at=None, expires_at=1780377531, failed_at=1780291133, finalizing_at=None, in_progress_at=None, metadata=None, model='gpt-4.1-mini-2025-04-14', output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=

In [2]:
BASE_DIR = Path.cwd().resolve().parents[1]
file_path = BASE_DIR / "data" / "rule_metadata_merged.json"

with open(file_path, "r", encoding="utf-8") as f:
    questions_data = json.load(f)

In [3]:
len(questions_data)

2

In [4]:
questions_data.keys()

dict_keys(['unique_rules', 'title_map'])

In [7]:
df = pd.DataFrame(questions_data['unique_rules'])
df[df['confidence'] == '애매']

,title,description,keywords,question,merged_questions,source_count,confidence,reason
3,산책,숲이 우거진 조용하고 평온한 분위기의 카페를 찾는 분들을 위한 추천입니다. 자연과 ...,"[숲속 카페, 조용한 카페, 한적한 분위기, 편안한 분위기, 힐링 카페, 아늑한 카...",숲이 우거진 조용한 카페 추천해 줄 수 있나요?,"[자연 속에서 여유롭게 쉴 수 있는 카페 어디 있을까요?, 나무가 많은 한적한 카페...",7,애매,자연·숲과 어우러진 산책 코스와 연계된 카페라는 인상을 줌
7,휴식,넓고 쾌적한 공간에서 편안하게 쉴 수 있는 카페를 찾는 분들을 위한 추천입니다.,"[넓고 쾌적한 공간, 편안한 분위기, 카페 공간_넓다, 매장_넓고 깔끔하다, 좌석_...",넓고 편안하게 쉴 수 있는 카페 추천해 주세요.,"[하루 종일 여유롭게 있을 수 있는 카페 추천 부탁해., 밤에 가기 좋은 분위기 있...",14,애매,몸과 마음을 쉬게 하는 공간이라는 점은 같지만 운동 후·더위 피하기 등 상황은 다름
9,카페분위기,편안하고 행복한 분위기에서 휴식과 대화를 즐기기 좋은 카페를 추천합니다.,"[가게, 행복한 분위기, 편안한 분위기, 카페 분위기, 조명 분위기, 포근한 분위기...",분위기 좋은 카페 추천해 줄 수 있어?,"[분위기 좋은 곳에서 커피 마시고 싶은데 어디가 괜찮을까요?, 마음까지 따뜻해지는 ...",22,애매,전반적인 분위기를 말하지만 지역·브랜드·시즌 등 세부 속성이 다름
10,여름,얼음이 듬뿍 들어간 시원한 음료가 유명한 카페를 찾는 분들을 위한 추천입니다. 더운...,"[얼음 낭낭, 시원한 분위기, 더운 날, 청량한 분위기, 음료, 더위, 테라스_시원...",얼음이 듬뿍 들어간 시원한 음료가 유명한 카페 추천해줄 수 있어?,"[얼음이 많이 들어가서 청량감 있는 카페 어디가 좋을까?, 더위 피하면서 여유롭게 ...",8,애매,더운 계절에 시원하게 머물 수 있는 카페라는 인상을 주지만 계절·온도 표현이 섞여 있음
12,외관,눈에 띄는 간판을 가진 분위기 좋은 카페를 찾는 분들을 위한 추천입니다.,"[간판, ATMOSPHERE, 카페 분위기, 가게_예쁘다, 인테리어_깔끔하다, 특별...",분위기 좋은 카페 중에서 눈에 띄는 간판을 가진 곳 추천해 줄 수 있어요?,[건물이 아름다운 카페를 찾고 있는데 추천해 주세요.],4,애매,카페의 바깥에서 보이는 모습과 간판 디자인을 포함한 외형을 의미
19,친절,서로를 배려하며 편안한 분위기에서 휴식과 대화를 즐길 수 있는 카페를 추천합니다.,"[배려, 편안한 분위기, 대화하기 좋은 카페, 편안한 카페, 아늑한 카페, 조용한 ...",분위기가 편안하고 서로를 배려하는 카페 어디 있을까요?,"[친구랑 조용히 담소 나누기 좋은 카페 어디 있을까?, 부부가 함께 가기 좋은 카페...",7,애매,사장님·직원의 친절함과 정감 있는 응대를 강조하지만 뉘앙스는 조금씩 다름
20,좌석많음,자리 공간이 넉넉하고 편안하여 여유롭게 머무를 수 있는 카페를 찾는 분들을 위한 추...,"[자리_넓다, 자리_넉넉하다, 좌석_넓다, 매장_넓다, 카페 공간_넉넉하다, 좌석_...",자리가 넉넉하고 편한 카페 추천해 줄래?,"[여유롭게 앉을 공간이 충분한 카페 알려 주세요., 자리 공간이 넉넉한 이디야 비슷...",22,애매,"좌석 수와 여유, 특정 좌석 위치까지 포함해 좌석 관련 편의성을 강조하지만 세부 뉘..."
23,감성소품,예쁜 컵으로 음료를 제공하는 감각적이고 아기자기한 인테리어가 돋보이는 카페를 추천합니다.,"[컵_이쁘다, 카페 인테리어_아기자기하다, 매장_예쁘다, 카페 인테리어_예쁘다, 인...",예쁜 컵으로 음료를 내오는 카페 추천해줄래?,[테이크아웃하기 편하고 인테리어 예쁜 카페 어디가 좋을까?],6,애매,소품·컵·포장 등 디테일이 예쁘다는 공통점이 있으나 대상이 다소 다름
24,모임,가족들이 모여 편안하고 예쁜 분위기에서 즐길 수 있는 넓고 쾌적한 카페를 추천합니다.,"[가족 모임_예쁘다, 가게_예쁘다, 매장_넓다, 카페 공간_넓고 예쁘다, 편안한 분...",가족들이 모이기 좋은 예쁜 카페 추천해 줄 수 있나요?,"[가족끼리 모이기 좋은 카페 추천해 주세요., 생일 축하하기 좋은 장소로 카페 어디...",27,애매,여럿이 모이는 상황을 공통적으로 말하지만 가족·지인·생일 등 목적은 다름
28,인테리어,기분 좋게 머물 수 있고 편안한 인테리어가 돋보이는 카페를 찾는 분들을 위한 추천입니다.,"[카페 인테리어_편안하다, 편안한 분위기, 아늑한 카페, 인테리어_아늑한 분위기, ...",기분 좋게 머물 수 있는 인테리어 좋은 카페 알려줄래?,"[예쁜 내부 꾸밈이 인상적인 카페 알려줄 수 있어?, 내부 꾸밈이 잘 되어 있는 카...",33,애매,모두 인테리어를 언급하지만 편안함·고급스러움·브랜드 스타일·독특함 등 방향이 조금씩 다름


In [9]:
import json


data = questions_data
print("병합 후 rule 수:", len(data["unique_rules"]))
print("title_map 수:", len(data["title_map"]))

# confidence 분포
from collections import Counter
conf = Counter(r["confidence"] for r in data["unique_rules"])
print("확신:", conf["확신"])
print("애매:", conf["애매"])

병합 후 rule 수: 125
title_map 수: 321
확신: 73
애매: 52


In [10]:
ambiguous = [r for r in data["unique_rules"] if r["confidence"] == "애매"]
for r in ambiguous:
    print(f"title: {r['title']}")
    print(f"reason: {r['reason']}")
    print()

title: 산책
reason: 자연·숲과 어우러진 산책 코스와 연계된 카페라는 인상을 줌

title: 휴식
reason: 몸과 마음을 쉬게 하는 공간이라는 점은 같지만 운동 후·더위 피하기 등 상황은 다름

title: 카페분위기
reason: 전반적인 분위기를 말하지만 지역·브랜드·시즌 등 세부 속성이 다름

title: 여름
reason: 더운 계절에 시원하게 머물 수 있는 카페라는 인상을 주지만 계절·온도 표현이 섞여 있음

title: 외관
reason: 카페의 바깥에서 보이는 모습과 간판 디자인을 포함한 외형을 의미

title: 친절
reason: 사장님·직원의 친절함과 정감 있는 응대를 강조하지만 뉘앙스는 조금씩 다름

title: 좌석많음
reason: 좌석 수와 여유, 특정 좌석 위치까지 포함해 좌석 관련 편의성을 강조하지만 세부 뉘앙스는 조금씩 다름

title: 감성소품
reason: 소품·컵·포장 등 디테일이 예쁘다는 공통점이 있으나 대상이 다소 다름

title: 모임
reason: 여럿이 모이는 상황을 공통적으로 말하지만 가족·지인·생일 등 목적은 다름

title: 인테리어
reason: 모두 인테리어를 언급하지만 편안함·고급스러움·브랜드 스타일·독특함 등 방향이 조금씩 다름

title: 콘센트
reason: 전자기기 사용과 충전을 위한 환경을 의미하지만 기기 종류는 다름

title: 예쁜카페
reason: 카페에 대한 긍정적 인상을 표현하나 ‘예쁜’은 미관, ‘좋은’은 종합 평가라 약간 차이가 있음

title: 감성
reason: 인테리어와 분위기가 감각적·세련되다는 미적 감성을 강조하나 세부 스타일은 다를 수 있음

title: 제로웨이스트
reason: 환경을 생각하는 소비·운영을 강조하지만 제로웨이스트는 더 강한 실천 개념

title: 디저트
reason: 디저트가 좋다는 공통점이 있으나 하나는 지역을 한정

title: 학생
reason: 초보자·어린이·학생 등 비교적 어린 층을 지칭하지만 대상 범위는 다름

ti

In [12]:
ambiguous = [r for r in data["unique_rules"] if r["confidence"] == "애매"]
for r in ambiguous:
    title = r["title"]
    # 이 title로 매핑된 원본들 찾기
    originals = [k for k, v in data["title_map"].items() if v == title and k != title]
    if originals:
        print(f"\n[{title}] ← {originals}")
        print(f"reason: {r['reason']}")


[산책] ← ['숲뷰', '숲속']
reason: 자연·숲과 어우러진 산책 코스와 연계된 카페라는 인상을 줌

[휴식] ← ['힐링', '시원한휴식', '더위쉼']
reason: 몸과 마음을 쉬게 하는 공간이라는 점은 같지만 운동 후·더위 피하기 등 상황은 다름

[카페분위기] ← ['분위기', '즐거운분위기', '근처분위기', '축제분위기', '세모디분위기', '이디야분위기']
reason: 전반적인 분위기를 말하지만 지역·브랜드·시즌 등 세부 속성이 다름

[여름] ← ['넓고시원', '시원한', '시원함', '더위']
reason: 더운 계절에 시원하게 머물 수 있는 카페라는 인상을 주지만 계절·온도 표현이 섞여 있음

[외관] ← ['간판']
reason: 카페의 바깥에서 보이는 모습과 간판 디자인을 포함한 외형을 의미

[친절] ← ['배려']
reason: 사장님·직원의 친절함과 정감 있는 응대를 강조하지만 뉘앙스는 조금씩 다름

[좌석많음] ← ['넉넉한자리', '넉넉한 자리', '넉넉한좌석', '자리여유', '자리넉넉', '자리', '자리편한', '창가자리', '조용한 좌석', '조용한 자리']
reason: 좌석 수와 여유, 특정 좌석 위치까지 포함해 좌석 관련 편의성을 강조하지만 세부 뉘앙스는 조금씩 다름

[감성소품] ← ['특별한소품', '컵디자인', '컵이쁜', '컵예쁨', '포장예쁨', '예쁜테이크아웃']
reason: 소품·컵·포장 등 디테일이 예쁘다는 공통점이 있으나 대상이 다소 다름

[모임] ← ['단체', '단체모임', '지인모임', '가족모임', '생일모임', '생일파티']
reason: 여럿이 모이는 상황을 공통적으로 말하지만 가족·지인·생일 등 목적은 다름

[인테리어] ← ['편안한 인테리어', '고급인테리어', '스타벅스인테리어', '독특한인테리어']
reason: 모두 인테리어를 언급하지만 편안함·고급스러움·브랜드 스타일·독특함 등 방향이 조금씩 다름

[콘센트] ← ['노트북', '아이패드', '충전']
reas

In [ ]:
"""
rule_metadata_merged.json 수동 패치 스크립트
"""
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

BASE_DIR    = Path(__file__).resolve().parents[2]
INPUT_PATH  = BASE_DIR / "data" / "rule_metadata_merged.json"
OUTPUT_PATH = BASE_DIR / "data" / "rule_metadata_merged.json"  # 덮어쓰기

with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)

rules     = data["unique_rules"]
title_map = data["title_map"]

# ── 헬퍼 ──────────────────────────────────────────────────────
def get_rule(title):
    for r in rules:
        if r["title"] == title:
            return r
    return None

def delete_rule(title):
    """rule 삭제 + title_map에서 해당 title로 매핑된 것도 제거"""
    global rules
    rules = [r for r in rules if r["title"] != title]
    # title_map에서 이 title로 매핑된 항목 제거
    keys_to_del = [k for k, v in title_map.items() if v == title]
    for k in keys_to_del:
        del title_map[k]

def merge_into(src_title, dst_title):
    """src rule을 dst rule에 keywords/merged_questions 합치고 src 삭제"""
    src = get_rule(src_title)
    dst = get_rule(dst_title)
    if not src or not dst:
        print(f"  ⚠️ merge_into 실패: {src_title} → {dst_title}")
        return
    # keywords 합집합
    seen = set(dst["keywords"])
    for kw in src["keywords"]:
        if kw not in seen:
            seen.add(kw)
            dst["keywords"].append(kw)
    # merged_questions 합치기
    dst["merged_questions"].extend(src.get("merged_questions", []))
    dst["merged_questions"].append(src["question"])
    # title_map 재매핑
    for k, v in title_map.items():
        if v == src_title:
            title_map[k] = dst_title
    delete_rule(src_title)

def remap_title_map(old_title, new_title):
    """title_map에서 old → new로 재매핑 (rule은 건드리지 않음)"""
    for k in title_map:
        if title_map[k] == old_title:
            title_map[k] = new_title

# ── 패치 시작 ─────────────────────────────────────────────────
print("🔧 패치 시작...")

# 1. 산책 삭제, 숲뷰/숲속 → 새 rule "숲속카페"로 합치기
#    (기존 산책 rule을 "숲속카페"로 rename 후 유지)
r = get_rule("산책")
if r:
    r["title"] = "숲속카페"
    remap_title_map("산책", "숲속카페")
    # 숲뷰, 숲속 title_map 재매핑
    for k in ["숲뷰", "숲속"]:
        title_map[k] = "숲속카페"
    print("  ✅ 산책 → 숲속카페 rename")

# 2. 여름 삭제 (넓고시원 → 좌석많음, 시원한/시원함 삭제)
for k in ["넓고시원"]:
    title_map[k] = "좌석많음"
for k in ["시원한", "시원함", "더위"]:
    if k in title_map:
        del title_map[k]
delete_rule("여름")
print("  ✅ 여름 삭제, 넓고시원 → 좌석많음")

# 3. 텀블러 → 제로웨이스트에서 분리 (title_map에서 삭제)
if "텀블러" in title_map:
    del title_map["텀블러"]
print("  ✅ 텀블러 title_map 삭제")

# 4. 간식 삭제 (요거트/우유옵션 포함)
for k in ["요거트", "우유옵션"]:
    if k in title_map:
        del title_map[k]
delete_rule("간식")
print("  ✅ 간식 삭제")

# 5. 상쾌한분위기 → 카페분위기로 통합
if "상쾌한분위기" in title_map:
    title_map["상쾌한분위기"] = "카페분위기"
print("  ✅ 상쾌한분위기 → 카페분위기")

# 6. 문화생활 삭제, 상영/영화 → 새 rule "영화"로
r = get_rule("문화생활")
if r:
    r["title"] = "영화"
    remap_title_map("문화생활", "영화")
    for k in ["상영", "영화"]:
        title_map[k] = "영화"
    print("  ✅ 문화생활 → 영화 rename")

# 7. 위치좋은 삭제 (위치/회사근처/고시촌 포함)
for k in ["위치", "회사근처", "고시촌", "위치좋은"]:
    if k in title_map:
        del title_map[k]
delete_rule("위치좋은")
delete_rule("위치좋은분위기")
print("  ✅ 위치좋은 삭제")

# 8. 특별한분위기 → 카페분위기로 통합
merge_into("특별한분위기", "카페분위기")
for k in ["특별한", "특별함"]:
    if k in title_map:
        del title_map[k]
print("  ✅ 특별한분위기 → 카페분위기 통합")

# 9. 대기/오픈런 삭제
for k in ["대기", "오픈런"]:
    if k in title_map:
        del title_map[k]
delete_rule("대기")
print("  ✅ 대기/오픈런 삭제")

# 10. 커피/마셔보기 삭제
for k in ["마셔보기", "커피"]:
    if k in title_map:
        del title_map[k]
delete_rule("커피")
print("  ✅ 커피/마셔보기 삭제")

# ── 저장 ──────────────────────────────────────────────────────
data["unique_rules"] = rules
data["title_map"]    = title_map

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"\n✅ 패치 완료!")
print(f"  rule 수: {len(rules)}개")
print(f"  title_map 수: {len(title_map)}개")